# 최소 scalar CLV M1~M5: Dunnhumby seed 42

학습기간의 사용자 historical CLV proxy `q_C=percentile(n×v)` 하나만 사용합니다. M2는 표현, M3는 그래프 엣지, M4는 BPR 양성행 가중에 같은 `q_C`를 직접 반영하고, M5는 세 개입을 결합합니다. 가격·가격구간·카테고리·별도 q_N/q_V 임베딩은 사용하지 않습니다. M1~M5 다섯 모형을 seed 42와 단일 음성 BPR로 새로 학습합니다. 이 실행은 이미 노출된 test의 사후 방향성 확인이므로 추가 튜닝이나 최종 확증에 사용하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '2933194dfcbe6d2e438aefaa24a5fb649ad5ec35'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(['git', 'clone', REPO_URL, str(repo)], text=True, capture_output=True)
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)


In [ ]:
import json
import torch
from lightgcn_clv_minimal_scalar_factorial_test import (
    configure_minimal_scalar_clv_test_run,
    preflight_summary,
    run_minimal_scalar_clv_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_minimal_scalar_clv_test_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_minimal_scalar_historical_clv_m1_m5_test_seed42_v1',
)
summary = preflight_summary(cfg)
assert cfg.seeds == (42,)
assert cfg.negative_count == 1
assert cfg.economic_dim == 1
assert len(summary['models']) == 5
assert summary['fixed']['validation_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['fixed']['new_item_task'] is True
assert summary['historical_clv_proxy']['future_clv_claim'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = run_minimal_scalar_clv_test(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) seed 42 M1~M5 전체 절대지표')
show(result_df)
print('2) M1·M2·M3·M4 각각을 기준으로 한 전체 지표 비교')
show(result_df.attrs['comparison'])
print('3) 구성요소별 M1 대비 차이와 M5의 단일모형 대비 차이')
show(result_df.attrs['interaction'])
print('4) 단일시드 기술적 판독')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
